# EMBED-CD — a change map from real satellite embeddings

This runs the whole pipeline on a small real area and **shows the result**: two years of
AlphaEarth embeddings, the change score between them, and the objects a cutoff produces.

`embed_cd` imports no QGIS, so it runs anywhere with **numpy, scipy, GDAL and matplotlib**.
QGIS's own Python already has all four — the least-effort way to run this:

```
# Windows
"C:\Program Files\QGIS <version>\bin\python-qgis.bat" -m pip install --user jupyterlab
"C:\Program Files\QGIS <version>\bin\python-qgis.bat" -m jupyterlab
# Linux / macOS
python3 -m pip install --user jupyterlab && python3 -m jupyterlab
```

Data: AlphaEarth Foundations Satellite Embedding V1, Google / Google DeepMind, CC BY 4.0.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

# Point at the repo checkout (skip if embed_cd is already importable).
REPO = os.path.abspath('..')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from embed_cd import source as SRC, score as S, head as H
from osgeo import gdal            # if this import fails, use QGIS's Python (see above)
print('GDAL', gdal.__version__, '- ready')

## 1 · Pick an area and two years

A few kilometres of Vancouver Island — active forestry, so there is real change to see.
The first run downloads a ~4 MB tile index once, then caches it.

In [ ]:
BBOX = (-125.34, 49.63, -125.24, 49.71)   # lon/lat; a small ROI
YEAR_A, YEAR_B = 2019, 2024

src = SRC.AlphaEarthSource()
all_tiles, both, partial = src.list_tiles(BBOX, YEAR_A, YEAR_B)
tile = all_tiles[0]                        # one 10 km tile is plenty to visualise
print(len(all_tiles), 'tile(s) cover this area; using one in', tile.crs)

## 2 · Fetch both years and score the change

`fetch` returns a `[H, W, 64]` cube of unit-length embeddings. `change_score` L2-normalises
each pixel's two years, takes the dot product, and remaps so **0 = identical, 1 = opposite**
— the AlphaEarth paper's own equation 8.

In [ ]:
cube_a, crs, _ = src.fetch(tile, YEAR_A)
cube_b, _,   _ = src.fetch(tile, YEAR_B)
print('embeddings:', cube_a.shape, '(H x W x 64)')

chg, cov = S.change_score(cube_a, cube_b)
valid = chg != S.NODATA

hist = S.histogram(chg)
cut = S.otsu_from_histogram(hist)          # the plugin's 'Auto' button
print('auto cutoff %.3f  ->  %.1f%% of valid pixels flagged'
      % (cut, 100 * S.fraction_above(hist, cut)))

## 3 · Turn the change into objects

Everything at or above the cutoff, grouped into connected blobs, with tiny specks dropped.
This is what the plugin's *Generate Embedded Vector Set* does.

In [ ]:
MIN_PX = 30                                # drop specks smaller than this
mask = valid & (chg >= cut)
labels, n = ndimage.label(mask)

sizes = ndimage.sum(np.ones_like(labels), labels, index=np.arange(1, n + 1))
keep = np.where(sizes >= MIN_PX)[0] + 1
objects = np.isin(labels, keep)
labels = np.where(objects, labels, 0)
print('%d object(s) at cutoff %.3f (after dropping specks)' % (len(keep), cut))

## 4 · See it

Before and after are a false colour built from three of the 64 embedding channels — not a
photo, but it shows the land structure. Then the change score, and the objects the cutoff
carved out.

In [ ]:
def falsecolour(cube, chans=(3, 7, 11)):
    """3 embedding channels -> an RGB image, each stretched 2-98%."""
    img = cube[..., list(chans)].astype(float)
    for i in range(3):
        lo, hi = np.percentile(img[..., i], (2, 98))
        img[..., i] = np.clip((img[..., i] - lo) / (hi - lo + 1e-9), 0, 1)
    return img

shown = np.where(valid, chg, np.nan)
obj_view = np.where(labels > 0, labels, np.nan)

fig, ax = plt.subplots(2, 2, figsize=(11, 10))
ax[0, 0].imshow(falsecolour(cube_a)); ax[0, 0].set_title('%d  (false colour)' % YEAR_A)
ax[0, 1].imshow(falsecolour(cube_b)); ax[0, 1].set_title('%d  (false colour)' % YEAR_B)

im = ax[1, 0].imshow(shown, cmap='magma', vmin=0, vmax=max(cut * 3, 0.05))
ax[1, 0].set_title('change score')
fig.colorbar(im, ax=ax[1, 0], fraction=0.046, pad=0.04)

ax[1, 1].imshow(falsecolour(cube_b))
ax[1, 1].imshow(obj_view, cmap='tab20', alpha=0.75, interpolation='nearest')
ax[1, 1].set_title('%d object(s) >= %.3f' % (len(keep), cut))
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); plt.show()

The score histogram, with the automatic cutoff. Almost everything sits near zero — most
of a scene does not change year to year — and the cutoff sits out on the tail.

In [ ]:
valid_scores = chg[valid]
plt.figure(figsize=(8, 3))
plt.hist(valid_scores, bins=80, range=(0, max(cut * 4, 0.1)), color='#166b6b')
plt.axvline(cut, color='#e8622a', lw=2, label='auto cutoff %.3f' % cut)
plt.xlabel('change score'); plt.ylabel('pixels'); plt.legend(); plt.tight_layout(); plt.show()

## 5 · Classify the objects (optional)

Give each object the mean of its *before* and *after* embeddings, label a couple by hand,
and let one detector per class label the rest. Here we seed two classes from the largest and
smallest objects — in the plugin you click real examples on the map.

In [ ]:
def object_vector(lbl):
    m = labels == lbl
    return np.concatenate([cube_a[m].mean(0), cube_b[m].mean(0)])   # [A | B]

ids = list(keep)
vecs = np.array([object_vector(i) for i in ids], np.float32)

if len(ids) >= 4:
    order = np.argsort([ (labels == i).sum() for i in ids ])[::-1]
    seed = {'larger change': vecs[order[:2]], 'smaller change': vecs[order[-2:]]}
    clf = H.fit_from_classes(seed, pool=vecs)
    pred, _ = clf.predict(vecs)

    palette = {c: plt.cm.Set1(i) for i, c in enumerate(list(clf.classes) + [H.UNKNOWN])}
    rgb = np.zeros(labels.shape + (4,))
    for i, p in zip(ids, pred):
        rgb[labels == i] = palette[str(p)]

    plt.figure(figsize=(6.5, 6))
    plt.imshow(falsecolour(cube_b))
    plt.imshow(np.where(labels[..., None] > 0, rgb, np.nan), alpha=0.8, interpolation='nearest')
    handles = [plt.Line2D([0], [0], marker='s', ls='', mfc=palette[c], mec='none',
                          label='%s (%d)' % (c, int((pred == c).sum())))
               for c in palette]
    plt.legend(handles=handles, loc='upper right', fontsize=8)
    plt.title('objects coloured by predicted class'); plt.xticks([]); plt.yticks([])
    plt.tight_layout(); plt.show()
else:
    print('need at least 4 objects to seed two classes; lower the cutoff or MIN_PX')

---

**What maps to what in the plugin**

| Here | Plugin |
|---|---|
| `change_score` | the change map layer |
| `otsu_from_histogram` | the **Auto** cutoff button |
| connected objects | **Generate Embedded Vector Set** |
| `fit_from_classes` / `predict` | click-to-label classification |

The one thing skipped for simplicity: the plugin reprojects every tile into one output grid,
so it mosaics many tiles seamlessly and writes GeoTIFFs. Here we stay on a single tile,
in memory, to keep the focus on seeing the result.